<a href="https://colab.research.google.com/github/anu0707/100-Days-Of-ML-Code/blob/master/notebooks/vakyansh_tts_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installing Dependencies

In [1]:
import os
!git clone https://github.com/Open-Speech-EkStep/vakyansh-tts
os.chdir('vakyansh-tts')
!bash install.sh
!python setup.py bdist_wheel
!pip install -e .
os.chdir('tts_infer')
!mkdir translit_models
os.chdir('translit_models')
!wget https://storage.googleapis.com/vakyaansh-open-models/translit_models/default_lineup.json
!mkdir hindi
os.chdir('hindi')
!wget https://storage.googleapis.com/vakyaansh-open-models/translit_models/hindi/hindi_transliteration.zip
!unzip hindi_transliteration

!wget https://storage.googleapis.com/vakyansh-open-models/tts/hindi/hi-IN/female_voice_0/glow.zip
!unzip glow.zip

!wget https://storage.googleapis.com/vakyansh-open-models/tts/hindi/hi-IN/female_voice_0/hifi.zip
!unzip hifi.zip

!rm glow.zip
!rm hifi.zip

os.chdir('/content/vakyansh-tts/')

Cloning into 'vakyansh-tts'...
remote: Enumerating objects: 1073, done.
remote: Counting objects: 100% (1073/1073), done.
remote: Compressing objects: 100% (411/411), done.
remote: Total 1073 (delta 650), reused 1070 (delta 649), pack-reused 0 (from 0)
Receiving objects: 100% (1073/1073), 402.46 KiB | 8.75 MiB/s, done.
Resolving deltas: 100% (650/650), done.
Processing /content/vakyansh-tts/src/glow_tts/monotonic_align
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 2
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 2
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Looking in links: https://download.pytorch.org/

## Inference Code

In [5]:
!pip install unidecode
!pip install pydload

In [6]:
from tts_infer.tts import TextToMel, MelToWav
from tts_infer.transliterate import XlitEngine
from tts_infer.num_to_word_on_sent import normalize_nums

import re
from scipy.io.wavfile import write
device = 'cpu'

text_to_mel = TextToMel(glow_model_dir='/content/vakyansh-tts/tts_infer/translit_models/hindi/glow_ckp', device=device)
mel_to_wav = MelToWav(hifi_model_dir='/content/vakyansh-tts/tts_infer/translit_models/hindi/hifi_ckp', device=device)

def translit(text, lang):
    reg = re.compile(r'[a-zA-Z]')
    engine = XlitEngine(lang)
    words = [engine.translit_word(word, topk=1)[lang][0] if reg.match(word) else word for word in text.split()]
    updated_sent = ' '.join(words)
    return updated_sent

def run_tts(text, lang):
    text = text.replace('।', '.') # only for hindi models
    text_num_to_word = normalize_nums(text, lang) # converting numbers to words in lang
    text_num_to_word_and_transliterated = translit(text_num_to_word, lang) # transliterating english words to lang

    mel = text_to_mel.generate_mel(text_num_to_word_and_transliterated)
    audio, sr = mel_to_wav.generate_wav(mel)
    write(filename='temp.wav', rate=sr, data=audio) # for saving wav file, if needed
    return (sr, audio)

/content/vakyansh-tts/tts_infer/translit_models/hindi/glow_ckp/G_250.pth


/content/vakyansh-tts/tts_infer/../src/glow_tts/modules.py:234: UserWarning: torch.qr is deprecated in favor of torch.linalg.qr and will be removed in a future PyTorch release.
The boolean parameter 'some' has been replaced with a string parameter 'mode'.
Q, R = torch.qr(A, some)
should be replaced with
Q, R = torch.linalg.qr(A, 'reduced' if some else 'complete') (Triggered internally at /pytorch/aten/src/ATen/native/BatchLinearAlgebra.cpp:2496.)
  w_init = torch.qr(torch.FloatTensor(self.n_split, self.n_split).normal_())[0]
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy.core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn mor

In [7]:
!grep -R "torch.load" -n /content/vakyansh-tts/src/glow_tts


/content/vakyansh-tts/src/glow_tts/hifi/utils.py:41:    checkpoint_dict = torch.load(filepath, map_location=device)
/content/vakyansh-tts/src/glow_tts/texttospeech.py:105:        state_dict_g = torch.load(checkpoint_path, map_location=self.device)
/content/vakyansh-tts/src/glow_tts/utils.py:20:    checkpoint_dict = torch.load(checkpoint_path, map_location="cpu")


In [ ]:
_, audio = run_tts('hello my name is harveen', 'hi')

## Results

In [ ]:
import IPython.display as ipd
ipd.Audio('temp.wav')